# Candidate Analysis: Similarities and Differences

This notebook analyzes political candidates from Frederiksberg Kommune based on their responses to policy questions.

## Objectives:
- Load and explore candidate data
- Analyze response patterns across candidates
- Compare candidates by party affiliation
- Identify consensus and divisive issues
- Visualize candidate similarities and differences

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
%matplotlib inline

## 1. Data Loading and Initial Exploration

In [ ]:
# Load the data
df = pd.read_csv('cleaned_01.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nFirst few rows:")
df.head()

In [ ]:
# Check data types and missing values
print("Data Info:")
df.info()
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Get unique candidates and parties
print(f"Total number of candidates: {df['candidate_id'].nunique()}")
print(f"\nNumber of unique parties: {df['party'].nunique()}")
print(f"\nParties: {sorted(df['party'].unique())}")
print(f"\nNumber of questions: {df['question_number'].nunique()}")

In [ ]:
# Create a summary of candidates by party
candidate_summary = df.groupby('party')['candidate_id'].nunique().sort_values(ascending=False)
print("Candidates per party:")
print(candidate_summary)

# Visualize
plt.figure(figsize=(10, 6))
candidate_summary.plot(kind='bar', color='steelblue')
plt.title('Number of Candidates by Party', fontsize=14, fontweight='bold')
plt.xlabel('Party', fontsize=12)
plt.ylabel('Number of Candidates', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Extract and Analyze Answer Patterns

We'll extract the actual answers (Enig/Uenig) from the answer_text field.

In [ ]:
# Function to extract answer position from answer_text
def extract_answer(text):
    """Extract Enig/Uenig answer from the answer_text field"""
    if pd.isna(text):
        return None
    
    # Look for patterns like "UenigXXXsvarEnig" where XXX is the candidate name
    if 'Uenig' in text and 'svar' in text and 'Enig' in text:
        # Find position of 'svar' - answer is between Uenig and svar
        parts = text.split('svar')
        if len(parts) >= 2:
            before_svar = parts[0]
            # Count occurrences to determine position
            # If only one 'Uenig' before 'svar', answer is 'Uenig'
            # If 'Enig' appears before 'svar', answer is 'Enig'
            if before_svar.count('Uenig') == 2:  # UenigXXXsvar pattern with Uenig answer
                return 'Uenig'
            elif 'Enig' in before_svar and before_svar.index('Enig') > before_svar.rindex('Uenig'):
                return 'Enig'
            else:
                return 'Uenig'
    return None

# Apply extraction
df['answer'] = df['answer_text'].apply(extract_answer)

# Check extraction results
print("Answer distribution:")
print(df['answer'].value_counts())
print(f"\nMissing answers: {df['answer'].isna().sum()}")

In [ ]:
# Create a pivot table: candidates x questions
# Encode answers as numeric: Enig=1, Uenig=-1, Missing=0
answer_mapping = {'Enig': 1, 'Uenig': -1}
df['answer_numeric'] = df['answer'].map(answer_mapping).fillna(0)

# Create pivot table
pivot_answers = df.pivot_table(
    index='candidate_id',
    columns='question_number',
    values='answer_numeric',
    fill_value=0
)

# Add candidate names and parties
candidate_info = df[['candidate_id', 'name', 'party']].drop_duplicates().set_index('candidate_id')
pivot_answers = pivot_answers.join(candidate_info)

print(f"Answer matrix shape: {pivot_answers.shape}")
pivot_answers.head()

## 3. Question-Level Analysis: Consensus vs. Divisive Issues

In [ ]:
# Analyze agreement levels for each question
question_stats = []

for q_num in sorted(df['question_number'].unique()):
    q_data = df[df['question_number'] == q_num]
    q_text = q_data['question_text'].iloc[0] if len(q_data) > 0 else ''
    
    answers = q_data['answer'].value_counts()
    total = len(q_data[q_data['answer'].notna()])
    
    if total > 0:
        enig_pct = (answers.get('Enig', 0) / total) * 100
        uenig_pct = (answers.get('Uenig', 0) / total) * 100
        
        # Calculate consensus score (how close to 100% or 0%)
        consensus = max(enig_pct, uenig_pct)
        
        question_stats.append({
            'question_number': q_num,
            'question_text': q_text[:100] + '...' if len(q_text) > 100 else q_text,
            'enig_pct': enig_pct,
            'uenig_pct': uenig_pct,
            'consensus_score': consensus,
            'total_responses': total
        })

question_df = pd.DataFrame(question_stats).sort_values('consensus_score', ascending=False)

print("\n=== MOST CONSENSUS (High Agreement) ===")
print(question_df.head(10)[['question_number', 'consensus_score', 'enig_pct', 'uenig_pct']])

print("\n=== MOST DIVISIVE (Split Opinions) ===")
print(question_df.tail(10)[['question_number', 'consensus_score', 'enig_pct', 'uenig_pct']])

In [ ]:
# Visualize consensus distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Histogram of consensus scores
axes[0].hist(question_df['consensus_score'], bins=20, color='teal', edgecolor='black')
axes[0].set_xlabel('Consensus Score (%)', fontsize=12)
axes[0].set_ylabel('Number of Questions', fontsize=12)
axes[0].set_title('Distribution of Consensus Scores', fontsize=14, fontweight='bold')
axes[0].axvline(x=50, color='red', linestyle='--', label='50% (Maximum Division)')
axes[0].legend()

# Top 10 most divisive questions
divisive = question_df.nsmallest(10, 'consensus_score')
y_pos = np.arange(len(divisive))
axes[1].barh(y_pos, divisive['consensus_score'], color='coral')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels([f"Q{q}" for q in divisive['question_number']], fontsize=10)
axes[1].set_xlabel('Consensus Score (%)', fontsize=12)
axes[1].set_title('Top 10 Most Divisive Questions', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 4. Party-Level Analysis

In [ ]:
# Calculate average position by party for each question
party_positions = df.groupby(['party', 'question_number'])['answer_numeric'].mean().reset_index()
party_pivot = party_positions.pivot(index='party', columns='question_number', values='answer_numeric')

print("Average party positions (1=Enig, -1=Uenig, 0=Mixed/No answer):")
party_pivot.head()

In [ ]:
# Visualize party positions as heatmap
plt.figure(figsize=(16, 8))
sns.heatmap(party_pivot, cmap='RdBu_r', center=0, vmin=-1, vmax=1, 
            cbar_kws={'label': 'Position (Red=Uenig, Blue=Enig)'},
            linewidths=0.5, linecolor='gray')
plt.title('Party Positions Across All Questions', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Question Number', fontsize=12)
plt.ylabel('Party', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate party similarity matrix
from scipy.spatial.distance import cosine

parties = party_pivot.index.tolist()
n_parties = len(parties)
similarity_matrix = np.zeros((n_parties, n_parties))

for i, party1 in enumerate(parties):
    for j, party2 in enumerate(parties):
        vec1 = party_pivot.loc[party1].values
        vec2 = party_pivot.loc[party2].values
        # Use correlation as similarity measure
        similarity = np.corrcoef(vec1, vec2)[0, 1]
        similarity_matrix[i, j] = similarity

similarity_df = pd.DataFrame(similarity_matrix, index=parties, columns=parties)

# Visualize party similarity
plt.figure(figsize=(10, 8))
sns.heatmap(similarity_df, annot=True, fmt='.2f', cmap='YlGnBu', 
            square=True, linewidths=1, cbar_kws={'label': 'Correlation'})
plt.title('Party Similarity Matrix\n(Based on Policy Positions)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Candidate Clustering and Similarity Analysis

In [ ]:
# Prepare data for clustering (exclude name and party columns)
X = pivot_answers.iloc[:, :-2].values  # All question columns
candidate_names = pivot_answers['name'].values
candidate_parties = pivot_answers['party'].values

# Perform PCA for dimensionality reduction and visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
# Visualize candidates in 2D PCA space, colored by party
plt.figure(figsize=(14, 10))

# Get unique parties and assign colors
unique_parties = np.unique(candidate_parties)
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_parties)))
party_colors = {party: colors[i] for i, party in enumerate(unique_parties)}

for party in unique_parties:
    mask = candidate_parties == party
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], 
               label=party, alpha=0.7, s=100, 
               color=party_colors[party], edgecolors='black', linewidth=0.5)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.title('Candidate Positions in Political Space\n(PCA Projection)', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Hierarchical clustering of candidates
from scipy.cluster.hierarchy import dendrogram, linkage

# Compute linkage
linkage_matrix = linkage(X, method='ward')

# Create dendrogram
plt.figure(figsize=(16, 10))
dendrogram(linkage_matrix, labels=candidate_names, leaf_font_size=8)
plt.title('Hierarchical Clustering of Candidates\n(Based on Policy Positions)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Candidate', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## 6. Find Most Similar and Different Candidate Pairs

In [ ]:
# Calculate pairwise distances between all candidates
from scipy.spatial.distance import pdist, squareform

distances = pdist(X, metric='euclidean')
distance_matrix = squareform(distances)

# Create DataFrame for easier analysis
distance_df = pd.DataFrame(distance_matrix, 
                          index=candidate_names, 
                          columns=candidate_names)

# Find most similar pairs (excluding self-comparisons)
similar_pairs = []
for i in range(len(candidate_names)):
    for j in range(i+1, len(candidate_names)):
        similar_pairs.append({
            'candidate1': candidate_names[i],
            'party1': candidate_parties[i],
            'candidate2': candidate_names[j],
            'party2': candidate_parties[j],
            'distance': distance_matrix[i, j]
        })

pairs_df = pd.DataFrame(similar_pairs).sort_values('distance')

print("\n=== 10 MOST SIMILAR CANDIDATE PAIRS ===")
print(pairs_df.head(10))

print("\n=== 10 MOST DIFFERENT CANDIDATE PAIRS ===")
print(pairs_df.tail(10))

## 7. Within-Party vs. Between-Party Variation

In [ ]:
# Analyze within-party vs between-party distances
pairs_df['same_party'] = pairs_df['party1'] == pairs_df['party2']

within_party = pairs_df[pairs_df['same_party']]['distance']
between_party = pairs_df[~pairs_df['same_party']]['distance']

print(f"Within-party average distance: {within_party.mean():.2f} (std: {within_party.std():.2f})")
print(f"Between-party average distance: {between_party.mean():.2f} (std: {between_party.std():.2f})")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Box plot
data_to_plot = [within_party, between_party]
axes[0].boxplot(data_to_plot, labels=['Within Party', 'Between Party'])
axes[0].set_ylabel('Distance', fontsize=12)
axes[0].set_title('Distribution of Candidate Distances', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Histogram
axes[1].hist(within_party, bins=30, alpha=0.6, label='Within Party', color='blue')
axes[1].hist(between_party, bins=30, alpha=0.6, label='Between Party', color='red')
axes[1].set_xlabel('Distance', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distance Distribution Comparison', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Individual Candidate Profiles

In [ ]:
# Function to display candidate profile
def show_candidate_profile(candidate_name):
    """Display detailed profile for a specific candidate"""
    candidate_data = df[df['name'] == candidate_name].copy()
    
    if len(candidate_data) == 0:
        print(f"Candidate '{candidate_name}' not found.")
        return
    
    party = candidate_data['party'].iloc[0]
    candidate_id = candidate_data['candidate_id'].iloc[0]
    
    print(f"\n{'='*80}")
    print(f"CANDIDATE PROFILE: {candidate_name}")
    print(f"Party: {party} | ID: {candidate_id}")
    print(f"{'='*80}\n")
    
    # Count answers
    answer_counts = candidate_data['answer'].value_counts()
    total_answered = len(candidate_data[candidate_data['answer'].notna()])
    
    print(f"Total questions answered: {total_answered}")
    print(f"Enig: {answer_counts.get('Enig', 0)} ({answer_counts.get('Enig', 0)/total_answered*100:.1f}%)")
    print(f"Uenig: {answer_counts.get('Uenig', 0)} ({answer_counts.get('Uenig', 0)/total_answered*100:.1f}%)")
    
    # Find most similar candidates from other parties
    candidate_idx = np.where(candidate_names == candidate_name)[0][0]
    distances_to_others = distance_matrix[candidate_idx]
    
    similar_candidates = []
    for i, (name, party_i, dist) in enumerate(zip(candidate_names, candidate_parties, distances_to_others)):
        if name != candidate_name and party_i != party:
            similar_candidates.append({'name': name, 'party': party_i, 'distance': dist})
    
    similar_df = pd.DataFrame(similar_candidates).sort_values('distance').head(5)
    
    print(f"\nMost similar candidates from other parties:")
    for idx, row in similar_df.iterrows():
        print(f"  - {row['name']} ({row['party']}) - Distance: {row['distance']:.2f}")

# Example: Show profile for first candidate
if len(candidate_names) > 0:
    show_candidate_profile(candidate_names[0])

In [ ]:
# Interactive: Choose a candidate to analyze
print("Available candidates:")
for i, name in enumerate(sorted(candidate_names), 1):
    party = candidate_parties[np.where(candidate_names == name)[0][0]]
    print(f"{i}. {name} ({party})")

# Uncomment and modify to analyze specific candidates:
# show_candidate_profile("Candidate Name Here")

## 9. Summary Statistics and Key Findings

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS SUMMARY")
print("="*80)

print(f"\n1. DATASET OVERVIEW")
print(f"   - Total candidates: {len(candidate_names)}")
print(f"   - Total parties: {len(unique_parties)}")
print(f"   - Total questions: {len(question_df)}")

print(f"\n2. CONSENSUS ANALYSIS")
high_consensus = question_df[question_df['consensus_score'] > 80]
low_consensus = question_df[question_df['consensus_score'] < 60]
print(f"   - High consensus questions (>80%): {len(high_consensus)}")
print(f"   - Divisive questions (<60%): {len(low_consensus)}")
print(f"   - Average consensus score: {question_df['consensus_score'].mean():.1f}%")

print(f"\n3. PARTY COHESION")
print(f"   - Average within-party distance: {within_party.mean():.2f}")
print(f"   - Average between-party distance: {between_party.mean():.2f}")
print(f"   - Ratio (between/within): {between_party.mean()/within_party.mean():.2f}x")

print(f"\n4. MOST ALIGNED PARTIES")
# Get top party pairs by similarity
party_pairs = []
for i, p1 in enumerate(parties):
    for j in range(i+1, len(parties)):
        p2 = parties[j]
        party_pairs.append({
            'party1': p1,
            'party2': p2,
            'similarity': similarity_df.loc[p1, p2]
        })
party_pairs_df = pd.DataFrame(party_pairs).sort_values('similarity', ascending=False)
for idx, row in party_pairs_df.head(3).iterrows():
    print(f"   - {row['party1']} ↔ {row['party2']}: {row['similarity']:.3f}")

print(f"\n5. MOST DIVERGENT PARTIES")
for idx, row in party_pairs_df.tail(3).iterrows():
    print(f"   - {row['party1']} ↔ {row['party2']}: {row['similarity']:.3f}")

print("\n" + "="*80)

## 10. Export Results

In [ ]:
# Save key results to CSV files
question_df.to_csv('question_consensus_analysis.csv', index=False)
pairs_df.to_csv('candidate_similarity_pairs.csv', index=False)
party_pivot.to_csv('party_positions.csv')
similarity_df.to_csv('party_similarity_matrix.csv')

print("Results exported to:")
print("  - question_consensus_analysis.csv")
print("  - candidate_similarity_pairs.csv")
print("  - party_positions.csv")
print("  - party_similarity_matrix.csv")